# End-to-end meaningful-stall experiment — theta300 to method fork

This is the single-file runner. Attach only CIFAR-100 and the dataset containing
`shared_seed1_epoch300.pt`, select **T4 x2**, and provide the Kaggle Secret
`github_token`.

The notebook keeps the theta300 LR unchanged, saves every exact trigger best,
and waits for 100 epochs without a significant +0.1 pp trigger improvement. It
then forks the saved
best state as `plateau_checkpoint.pt`, never the later degraded state,
into E-driven O, O-only, and matched-horizon Bypass. The already-observed
100-epoch Vanilla stall trajectory is reused as the control rather than trained
twice. There is no need to save and reattach a Phase-1 dataset between stages.

Epoch 500 remains a review boundary. If no stall is found, Phase 2 is not run;
both `checkpoint_latest.pt` and `checkpoint_best.pt` are retained. Attach that
notebook output, increase `MAX_EPOCH`, and rerun to resume Phase 1 exactly.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
OUTPUT = Path("/kaggle/working/vanilla_best_stall_theta300_v2")
MAX_EPOCH = 500  # raise this on a resumed run if plateau_found is false

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                        url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, "main")
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO,
               env=test_env, check=True)
import torch
if torch.cuda.device_count() < 1:
    raise RuntimeError("Select a Kaggle GPU accelerator")
cifar_dirs = sorted({path.parent.resolve()
                     for path in Path("/kaggle/input").rglob("cifar-100-python")})
if not cifar_dirs: raise FileNotFoundError("Attach CIFAR-100")
DATA_ROOT = cifar_dirs[0]


In [ ]:
if torch.cuda.device_count() != 2:
    raise RuntimeError(
        f"Select the Kaggle T4 x2 accelerator; found {torch.cuda.device_count()} GPU(s)")


In [ ]:
def materialize(path, index, prefix):
    if path.is_file(): return path
    pickles = list(path.rglob("data.pkl"))
    if len(pickles) != 1: return None
    root = pickles[0].parent
    target = OUTPUT / "repacked_input" / f"{prefix}_{index}.pt"
    target.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_STORED) as archive:
        for record in sorted(root.rglob("*")):
            if record.is_file():
                relative = record.relative_to(root).as_posix()
                archive.writestr(f"{prefix}/{relative}", record.read_bytes())
    return target

forks = []
for index, raw in enumerate(Path("/kaggle/input").rglob("shared_seed1_epoch300.pt")):
    path = materialize(raw, index, "shared_seed1_epoch300")
    if path is None: continue
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if payload.get("kind") == "shared_fork_checkpoint" and payload.get("epoch") == 300:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        forks.append((path, digest))
if not forks: raise FileNotFoundError("Attach shared_seed1_epoch300.pt")
if len({digest for _, digest in forks}) != 1:
    raise RuntimeError("Multiple different theta300 checkpoints attached")
THETA300, THETA300_HASH = forks[0]

# Optional continuation from an earlier compatible Phase-1 output.
resume_payload = None
for index, raw in enumerate(Path("/kaggle/input").rglob("checkpoint_latest.pt")):
    path = materialize(raw, index, "checkpoint_latest")
    if path is None: continue
    try: payload = torch.load(path, map_location="cpu", weights_only=False)
    except Exception: continue
    if (payload.get("kind") == "vanilla_convergence_progress" and
            payload.get("protocol", {}).get("source_checkpoint_hash") == THETA300_HASH and
            payload.get("protocol", {}).get("schedule_id") == "constant-theta300-lr-best-stall-v1"):
        shutil.copy2(path, OUTPUT / "checkpoint_latest.pt")
        resume_payload = payload
        print("Restored Phase-1 progress at epoch", payload["epoch"])
        break
if resume_payload is not None:
    matching_best = []
    for index, raw in enumerate(Path("/kaggle/input").rglob("checkpoint_best.pt")):
        path = materialize(raw, index, "checkpoint_best")
        if path is None: continue
        try: payload = torch.load(path, map_location="cpu", weights_only=False)
        except Exception: continue
        if (payload.get("kind") == "vanilla_best_checkpoint" and
                payload.get("protocol") == resume_payload.get("protocol") and
                int(payload.get("epoch", -1)) ==
                int(resume_payload["best_stall_detector"]["best_epoch"])):
            matching_best.append(path)
    if len(matching_best) != 1:
        raise RuntimeError("Resume requires exactly one matching checkpoint_best.pt")
    shutil.copy2(matching_best[0], OUTPUT / "checkpoint_best.pt")
    print("Restored best checkpoint at epoch",
          resume_payload["best_stall_detector"]["best_epoch"])
print("theta300:", THETA300, THETA300_HASH)


## Phase 1 — Vanilla until no-new-best stall

Only GPU0 is needed during convergence search. The second T4 becomes active as
soon as Phase 2 starts.


In [ ]:
command = [
    sys.executable, "-m", "experiments.run_vanilla_to_plateau",
    "--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
    "--fork-checkpoint", str(THETA300),
    "--fork-checkpoint-hash", THETA300_HASH,
    "--output", str(OUTPUT), "--max-epoch", str(MAX_EPOCH),
    "--seed", "1", "--batch-size", "64", "--validation-samples", "5000",
    "--trigger-samples", "2000",
    "--no-new-best-patience", "100",
    "--best-min-gain", "0.0",
    "--significant-min-gain", "0.001"]
env = os.environ.copy()
env.update(CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1",
           PYTHONPATH=RUNTIME_PYTHONPATH)
process = subprocess.Popen(command, cwd=REPO, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with (OUTPUT / "run.log").open("a", buffering=1) as log:
    for line in process.stdout:
        log.write(line); print(line, end="", flush=True)
if process.wait(): raise RuntimeError("Vanilla convergence search failed")


In [ ]:
phase1_result = json.loads((OUTPUT / "result.json").read_text())
print(json.dumps({key: phase1_result[key] for key in (
    "status", "plateau_found", "plateau_epoch", "best_epoch",
    "stall_detected_epoch", "epochs_without_improvement", "review_epoch_reached",
    "final_validation_accuracy", "best_validation_accuracy",
    "final_validation_loss", "plateau_checkpoint")}, indent=2))

PHASE1_OUTPUT = OUTPUT
if not phase1_result["plateau_found"]:
    archive = shutil.make_archive(str(PHASE1_OUTPUT), "gztar", root_dir=PHASE1_OUTPUT)
    print("NO CONVERGENCE CLAIM. Resume this notebook with the current output.")
    print("Phase-1 archive:", archive)
    raise RuntimeError(
        "Review horizon reached without a no-new-best stall; Phase 2 was intentionally skipped")

PLATEAU_CHECKPOINT = Path(phase1_result["plateau_checkpoint"])
if not PLATEAU_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Phase 1 reported a missing checkpoint: {PLATEAU_CHECKPOINT}")
plateau_payload = torch.load(
    PLATEAU_CHECKPOINT, map_location="cpu", weights_only=False)
if plateau_payload.get("kind") != "plateau_fork_checkpoint":
    raise RuntimeError("Phase 1 did not produce a plateau fork checkpoint")
PLATEAU_EPOCH = int(plateau_payload["epoch"])
PLATEAU_HASH = hashlib.sha256(PLATEAU_CHECKPOINT.read_bytes()).hexdigest()
PLATEAU_PAYLOAD = plateau_payload
VANILLA_CONTROL = dict(PLATEAU_PAYLOAD["vanilla_control"])
if VANILLA_CONTROL["post_fork_epochs"] != 100:
    raise RuntimeError("Phase 1 must contain exactly 100 matched Vanilla epochs")
print("theta_P ready:", PLATEAU_EPOCH, PLATEAU_CHECKPOINT, PLATEAU_HASH)


## Phase 2 — Fixed T4 x2 schedule

GPU0 runs E-driven O only. GPU1 runs Bypass then fresh O-only. Every method is
required to report
the exact hash produced by Phase 1.


In [ ]:
import threading

OUTPUT = Path("/kaggle/working/plateau_end_to_end_three_methods_100ep_v4")
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
def base_args(output):
    return [
        "--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
        "--plateau-checkpoint", str(PLATEAU_CHECKPOINT),
        "--plateau-checkpoint-hash", PLATEAU_HASH,
        "--output", str(output), "--post-fork-epochs", "100",
        "--seed", "1", "--batch-size", "64", "--rank", "4",
        "--opt1-epochs", "70", "--max-opt2-epochs", "30",
        "--probe-epsilon", "0.05", "--where-batches", "3",
        "--retrigger-patience", "10",
        "--significant-improvement", "0.001",
        "--gamma-increase-opt2-epoch", "15",
        "--gamma-post-increase-multiplier", "2.0",
        "--line-search-scales", "0.0125,0.025,0.05"]

commands = {
    name: [sys.executable, "-m", "experiments.run_plateau_fork",
           "--method", name] + base_args(OUTPUT / name)
    for name in ("bypass", "ours_e_driven_o", "o_projection_only")}

def stream(name, process, log):
    for line in process.stdout:
        log.write(line); log.flush()
        print(f"[{name}] {line}", end="", flush=True)

def launch(gpu, name):
    out = OUTPUT / name; out.mkdir(parents=True, exist_ok=True)
    log = (out / "run.log").open("a", buffering=1)
    env = os.environ.copy()
    env.update(CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONPATH=RUNTIME_PYTHONPATH)
    process = subprocess.Popen(
        commands[name], cwd=REPO, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    thread = threading.Thread(target=stream, args=(name, process, log), daemon=True)
    thread.start()
    print(f"GPU{gpu}: started {name}, pid={process.pid}")
    return name, process, thread, log

def finish(job):
    name, process, thread, log = job
    code = process.wait()
    thread.join(); log.close()
    print(name, "exit=", code)
    return name, code

print("GPU0: E-driven O only")
print("GPU1: Bypass -> fresh O-only")
ours_job = launch(0, "ours_e_driven_o")
bypass_job = launch(1, "bypass")
bypass_status = finish(bypass_job)
if bypass_status[1]: raise RuntimeError(f"failed arm: {bypass_status}")
o_only_job = launch(1, "o_projection_only")
statuses = [finish(ours_job), finish(o_only_job)]
failures = [status for status in statuses if status[1]]
if failures: raise RuntimeError(f"failed arms: {failures}")


In [ ]:
results = {name: json.loads((OUTPUT / name / "result.json").read_text())
           for name in ("bypass", "ours_e_driven_o", "o_projection_only")}
for name, result in results.items():
    if result["plateau_checkpoint_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_P")
    if result["theta_best_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_best")
    if result["post_fork_epochs"] != 100:
        raise RuntimeError(f"{name} did not complete 100 epochs")
    if not Path(result["best_checkpoint"]).is_file():
        raise RuntimeError(f"{name} did not save checkpoint_best.pt")
VANILLA_CONTROL["plateau_checkpoint_hash"] = PLATEAU_HASH
VANILLA_CONTROL["theta_best_hash"] = PLATEAU_HASH
VANILLA_CONTROL["best_checkpoint"] = str(PLATEAU_CHECKPOINT)
results["vanilla"] = VANILLA_CONTROL
summary = {
    "plateau_epoch": PLATEAU_EPOCH,
    "plateau_checkpoint_hash": PLATEAU_HASH,
    "theta_best_hash": PLATEAU_HASH,
    "post_fork_epochs": 100, "official_test_used": False,
    "phase1_vanilla_control": VANILLA_CONTROL,
    "results": {name: {key: result.get(key) for key in (
        "fork_validation_accuracy", "fork_validation_loss",
        "theta_P_validation_accuracy", "theta_P_validation_loss",
        "meaningful_best_validation_accuracy",
        "fork_trigger_accuracy", "fork_trigger_loss",
        "exact_best_trigger_accuracy", "significant_best_trigger_accuracy",
        "final_validation_accuracy", "best_validation_accuracy",
        "final_validation_loss", "best_validation_loss",
        "validation_accuracy_delta", "best_validation_accuracy_delta",
        "epochs_to_best", "training_seconds", "peak_gpu_memory",
        "peak_train_params", "deploy_params", "time_spent_expanded_seconds",
        "bypass_completed", "contraction_at_projection",
        "opt1_epochs", "opt2_epochs", "train3_epochs",
        "gamma_increase_opt2_epoch", "gamma_post_increase_multiplier",
        "projection_loss_jump", "best_checkpoint", "intervention_count",
        "correction_application_count", "correction_application_rate",
        "intervention_seconds", "rollback_count", "retrigger_patience",
        "interventions")}
        for name, result in results.items()},
    "bypass_comparison_status": (
        "completed" if results["bypass"]["bypass_completed"] else
        "budget_exhausted_before_contraction_accuracy_is_diagnostic"),
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2, sort_keys=True))
print("\nmethod                  fork_acc   best_acc  final_acc  epoch_best")
for name in ("vanilla", "ours_e_driven_o", "bypass", "o_projection_only"):
    row = results[name]
    print(f"{name:23s} {row['fork_validation_accuracy']:.4f}     "
          f"{row['best_validation_accuracy']:.4f}    "
          f"{row['final_validation_accuracy']:.4f}    "
          f"{row['epochs_to_best']:>4}")
archive = shutil.make_archive(str(OUTPUT), "gztar", root_dir=OUTPUT)
print("Archive:", archive)
